# Základy testování (Testing Fundamentals)

Tento notebook slouží jako úvod do **testování software** v Pythonu.
Naučíte se:

- Co je testování a proč je důležité
- Jak funguje klíčové slovo `assert`
- Jak psát a spouštět testy pomocí knihovny **pytest**
- Jak přemýšlet o **hraničních případech** (edge cases)
- Jak vypadá vzor **Arrange-Act-Assert**

**Prerekvizity:** funkce, základy OOP, práce se soubory, knihovny a pip

---

## 1. Proč testujeme?

Představte si, že napíšete funkci, která počítá průměr ze seznamu čísel.
Spustíte ji jednou, výsledek vypadá dobře, odevzdáte. Ale co když:

- někdo předá **prázdný seznam**?
- vstup obsahuje **záporná čísla**?
- funkce je součástí většího projektu a za měsíc ji **někdo změní**?

### Proč ne prostě `print()` a zkontroluju očima?

Možná si říkáte: *"Já si prostě přidám `print(výsledek)` a podívám se, jestli to vypadá správně."*
To funguje jednou. Ale:

| Print debugging | Automatické testy |
|---|---|
| Musíte **ručně** kontrolovat výstup pokaždé | Spustíte jedním příkazem, výsledek OK/FAIL |
| Po opravě musíte **znovu** ručně ověřit vše | Testy ověří **všechno** automaticky za sekundy |
| Při 50 funkcích = 50× print + oční kontrola | Při 50 funkcích = 1× `pytest` |
| Nemůžete sdílet s kolegou | Kolega spustí testy a ví, že kód funguje |

**Testování** znamená napsat kód, který automaticky ověřuje, že se váš program chová správně. Testy:

1. **Odhalí chyby dříve** - než je najde uživatel (nebo učitel)
2. **Slouží jako dokumentace** - test ukazuje, jak se funkce má používat
3. **Dávají odvahu měnit kód** - po každé změně spustíte testy a víte, jestli jste něco nerozbili

### Testování v době AI

> **V době AI je testování ještě důležitější.** Pokud vám kód (nebo jeho část) vygeneruje AI,
> testy jsou váš hlavní nástroj, jak ověřit, že výsledek skutečně dělá to, co má.
>
> AI generátory kódu (jako GitHub Copilot nebo ChatGPT) jsou stále lepší, ale:
>
> - **Nevědí přesně, co chcete** — mohou nepochopit hraniční případy
> - **Mohou produkovat kód s jemnými chybami** — kód vypadá správně, ale nefunguje ve všech situacích
> - **Neumí ověřit samy sebe** — to musíte udělat vy, pomocí testů
>
> Schopnost napsat dobré testy je proto **dovednost, která ve věku AI roste na ceně**, ne klesá.

---

## 2. Klíčové slovo `assert`

Python má vestavěné klíčové slovo `assert`, které funguje jako **kontrolní výraz**.
Zápis je jednoduchý:

```python
assert podmínka, "volitelná zpráva při selhání"
```

- Pokud je `podmínka` pravdivá (`True`), nic se nestane - program pokračuje dál.
- Pokud je `podmínka` nepravdivá (`False`), Python vyhodí chybu `AssertionError`.

Pojďme si to vyzkoušet.

In [ ]:
# Nejdříve si definujeme jednoduchou funkci, kterou budeme testovat
def add(a, b):
    """Return the sum of two numbers."""
    return a + b

# Assert projde - podmínka je True, nic se nestane
assert add(2, 3) == 5
print("Test prošel: add(2, 3) == 5")

In [ ]:
# Můžeme řetězit více assertů za sebou
assert add(0, 0) == 0
assert add(-1, 1) == 0
assert add(100, 200) == 300
print("Všechny testy prošly!")

### Co se stane, když assert selže?

Pokud je podmínka `False`, Python zastaví program a vyhodí `AssertionError`.
Volitelná zpráva nám pomůže pochopit, **co přesně selhalo**.

In [ ]:
# Tohle selže - úmyslně špatný expected výsledek
assert add(2, 2) == 5, f"Očekáváno 5, ale dostali jsme {add(2, 2)}"

Vidíte `AssertionError` se zprávou, kterou jsme zadali. Přesně tak funguje i pytest - jen to za nás hezky zformátuje.

### Další typy porovnání v assertech

Assert umí pracovat s libovolnou podmínkou, nejen `==`:

In [ ]:
# Porovnání rovnosti
assert add(1, 2) == 3

# Porovnání nerovnosti
assert add(1, 2) != 0

# Větší / menší
assert add(10, 20) > 25
assert add(1, 1) <= 2

# Kontrola typu
assert isinstance(add(1, 2), int)

# Kontrola, že hodnota je v seznamu
assert add(1, 2) in [1, 2, 3, 4, 5]

# Kontrola pravdivosti (truthy)
assert add(1, 0)  # nenulové číslo je truthy

print("Všechny asserty prošly!")

> **Poznámka:** Funkci `add` jsme zde definovali přímo v notebooku pro rychlou ukázku.
> V další sekci ji přesuneme do samostatného souboru — tak se to dělá v reálných projektech.

---

## 3. Pytest - testovací framework

Klíčové slovo `assert` je užitečné, ale pro reálné projekty potřebujeme víc:

- **Automatické spouštění** všech testů najednou
- **Přehledný výstup** - co prošlo, co selhalo, a proč
- **Organizaci testů** do souborů a funkcí

K tomu slouží **pytest** - nejpopulárnější testovací framework pro Python.

### Instalace

Pytest je externí knihovna (vzpomeňte si na hodinu o knihovnách a pip):

In [ ]:
# Instalace pytestu (stačí spustit jednou)
!pip install pytest -q

### Jak pytest funguje?

Pytest pracuje se soubory a funkcemi. Pravidla jsou jednoduchá:

| Pravidlo | Příklad |
|---|---|
| Testovací soubor začíná na `test_` | `test_calculator.py` |
| Testovací funkce začíná na `test_` | `def test_add():` |
| Uvnitř funkce používáme `assert` | `assert add(2, 3) == 5` |

Pytest automaticky najde všechny takové soubory a funkce a spustí je.

### Vytvoříme si první testovaný modul

Pomocí `%%writefile` zapíšeme Python soubor přímo z notebooku.
V praxi byste tyto soubory psali ve svém editoru / IDE.

In [ ]:
%%writefile calculator.py
"""Simple calculator module for testing demonstration."""


def add(a: float, b: float) -> float:
    """Return the sum of two numbers."""
    return a + b


def subtract(a: float, b: float) -> float:
    """Return the difference of two numbers."""
    return a - b


def multiply(a: float, b: float) -> float:
    """Return the product of two numbers."""
    return a * b


def divide(a: float, b: float) -> float:
    """Return the division of a by b. Raises ValueError if b is zero."""
    if b == 0:
        raise ValueError("Cannot divide by zero!")
    return a / b

Teď napíšeme testovací soubor. Všimněte si:
- Název souboru začíná na `test_`
- Importujeme funkce z našeho modulu `calculator`
- Každá testovací funkce začíná na `test_`
- Uvnitř používáme obyčejné `assert`

In [ ]:
%%writefile test_calculator.py
"""Tests for the calculator module."""
from calculator import add, subtract, multiply, divide


# -- Tests for add --

def test_add_positive_numbers():
    assert add(2, 3) == 5

def test_add_negative_numbers():
    assert add(-1, -1) == -2

def test_add_zero():
    assert add(0, 0) == 0


# -- Tests for subtract --

def test_subtract_basic():
    assert subtract(10, 4) == 6

def test_subtract_negative_result():
    assert subtract(3, 7) == -4


# -- Tests for multiply --

def test_multiply_basic():
    assert multiply(3, 4) == 12

def test_multiply_by_zero():
    assert multiply(5, 0) == 0


# -- Tests for divide --

def test_divide_basic():
    assert divide(10, 2) == 5.0

def test_divide_decimal_result():
    assert divide(7, 2) == 3.5

### Spuštění testů

Testy spustíme příkazem `pytest` z terminálu. Přepínač `-v` (verbose) zobrazí detail každého testu:

In [ ]:
!pytest test_calculator.py -v

### Čtení výstupu pytestu

Ve výstupu vidíte:
- **PASSED** (zelená tečka) - test prošel
- Název souboru a funkce u každého testu
- Souhrn na konci: kolik testů prošlo, kolik selhalo

Zkusme teď přidat test, který **selže**, abychom viděli, jak pytest zobrazuje chyby:

In [ ]:
%%writefile test_calculator_fail.py
"""Test file with an intentionally failing test."""
from calculator import add


def test_add_correct():
    assert add(2, 3) == 5


def test_add_wrong_expectation():
    # Tento test SELŽE - úmyslně špatná očekávaná hodnota
    assert add(2, 2) == 5

In [ ]:
!pytest test_calculator_fail.py -v

Všimněte si, jak pytest u selhávajícího testu zobrazí:
- **Řádek, kde assert selhal**
- **Skutečnou hodnotu** vs. **očekávanou hodnotu** (např. `4 == 5`)
- Přehledný souhrn: `1 passed, 1 failed`

Toto je jedna z hlavních výhod pytestu oproti holému `assert` - **nemusíte psát chybové zprávy ručně**, pytest je vygeneruje za vás.

---

## 4. Vzor Arrange-Act-Assert (AAA)

Při psaní testů se osvědčil jednoduchý vzor se třemi kroky:

1. **Arrange** (Připrav) - Připravíme si vstupní data a očekávané výsledky
2. **Act** (Proveď) - Zavoláme testovanou funkci
3. **Assert** (Ověř) - Zkontrolujeme, že výsledek odpovídá očekávání

Tento vzor pomáhá udržet testy **přehledné a čitelné**. Podívejme se na příklad
s naší kalkulačkou:

In [ ]:
%%writefile test_calculator_aaa.py
"""Tests demonstrating the Arrange-Act-Assert pattern."""
from calculator import add, subtract, multiply, divide


def test_add_exam_scores():
    # ARRANGE - připravíme si data
    score1 = 85
    score2 = 92
    expected_total = 177

    # ACT - zavoláme testovanou funkci
    result = add(score1, score2)

    # ASSERT - ověříme výsledek
    assert result == expected_total


def test_calculate_change():
    # ARRANGE
    price = 350
    paid = 500
    expected_change = 150

    # ACT
    change = subtract(paid, price)

    # ASSERT
    assert change == expected_change


def test_total_price_with_quantity():
    # ARRANGE
    unit_price = 49.90
    quantity = 3

    # ACT
    total = multiply(unit_price, quantity)

    # ASSERT
    assert total == 149.70

In [ ]:
!pytest test_calculator_aaa.py -v

Všimněte si, jak komentáře `# ARRANGE`, `# ACT`, `# ASSERT` pomáhají vizuálně
oddělit tři fáze testu. Nemusíte je psát pokaždé, ale zpočátku vám pomohou
udržet testy přehledné.

> **Tip:** Pokud je váš test delší než ~10 řádků, AAA komentáře výrazně
> zlepší čitelnost i pro ostatní (nebo pro vás za měsíc).

---

## 5. Hraniční případy (Edge Cases)

Toto je pravděpodobně **nejdůležitější dovednost** spojená s testováním.
Napsat test na "normální" případ je snadné. Skutečná hodnota testování je v tom,
že nás nutí přemýšlet o **situacích, na které bychom jinak zapomněli**.

### Jak přemýšlet o edge casech?

Při testování jakékoliv funkce se ptejte:

| Otázka | Příklad |
|---|---|
| Co když je vstup **prázdný**? | Prázdný seznam, prázdný řetězec |
| Co když je vstup **nulový**? | `0`, `0.0` |
| Co když je vstup **záporný**? | `-1`, `-100` |
| Co když je vstup **velmi velký**? | `10**18`, milion prvků |
| Co když je vstup **neočekávaného typu**? | String místo čísla |
| Co když je vstup na **hranici** podmínky? | Přesně `0`, přesně `100` |

### Ukázka: testování funkce pro výpočet průměru

Napišme funkci `average` a pokusme se ji důkladně otestovat:

In [ ]:
%%writefile stats.py
"""Simple statistics module."""


def average(numbers: list[float]) -> float:
    """Return the arithmetic mean of a list of numbers.
    
    Raises ValueError if the list is empty.
    """
    if not numbers:
        raise ValueError("Cannot compute average of empty list")
    return sum(numbers) / len(numbers)

In [ ]:
%%writefile test_stats.py
"""Thorough tests for the stats module - demonstrating edge case thinking."""
import pytest
from stats import average


# --- Normální (happy path) případy ---

def test_average_basic():
    """Průměr z několika kladných čísel."""
    assert average([1, 2, 3, 4, 5]) == 3.0


def test_average_two_numbers():
    """Průměr ze dvou čísel."""
    assert average([10, 20]) == 15.0


# --- Hraniční případy ---

def test_average_single_element():
    """Průměr ze seznamu s jedním prvkem = ten prvek."""
    assert average([42]) == 42.0


def test_average_negative_numbers():
    """Průměr ze záporných čísel."""
    assert average([-1, -2, -3]) == -2.0


def test_average_mixed_positive_negative():
    """Průměr ze směsi kladných a záporných čísel."""
    assert average([-10, 10]) == 0.0


def test_average_with_zeros():
    """Průměr ze samých nul."""
    assert average([0, 0, 0]) == 0.0


def test_average_decimal_numbers():
    """Průměr z desetinných čísel."""
    result = average([1.5, 2.5])
    assert result == 2.0


def test_average_large_numbers():
    """Průměr z velkých čísel - ověříme, že nedojde k přetečení."""
    assert average([10**9, 10**9]) == 10**9


# --- Chybové případy ---

def test_average_empty_list_raises():
    """Prázdný seznam musí vyhodit ValueError."""
    with pytest.raises(ValueError, match="empty"):
        average([])

In [ ]:
!pytest test_stats.py -v

Všimněte si, jak jsou testy **organizované** do skupin:
- **Happy path** - normální, očekávané použití
- **Hraniční případy** - jeden prvek, nuly, záporná čísla, velká čísla
- **Chybové případy** - co se stane při neplatném vstupu

Toto členění je dobrý zvyk - pomáhá ujistit se, že na nic nezapomenete.

### Pozor na desetinná čísla (floating point)

Při testování výpočtů s desetinnými čísly narazíte na záludný problém.
Podívejte se na následující příklad:

In [ ]:
# Tohle by mělo být True, ne?
print(f"0.1 + 0.2 = {0.1 + 0.2}")
print(f"0.1 + 0.2 == 0.3? {0.1 + 0.2 == 0.3}")  # False!

**Proč?** Počítače ukládají desetinná čísla v **binárním formátu** (IEEE 754 floating point).
Některá čísla, která vypadají jednoduše v desítkové soustavě (jako `0.1`), 
**nelze přesně reprezentovat** v binární soustavě - podobně jako 1/3 = 0.333... 
nelze přesně zapsat v desítkové soustavě.

Výsledek `0.1 + 0.2` proto není přesně `0.3`, ale `0.30000000000000004`.

### Jak to řešit v testech?

**Špatně** - přímé porovnání selže:

In [ ]:
# ŠPATNĚ - tento assert selže!
assert 0.1 + 0.2 == 0.3, f"Dostali jsme {0.1 + 0.2}, ne 0.3"

**Správně** - použijeme `pytest.approx`, který porovnává s malou tolerancí:

In [ ]:
import pytest

# SPRÁVNĚ - pytest.approx porovnává s malou tolerancí (default: 1e-6)
assert 0.1 + 0.2 == pytest.approx(0.3)
print("Test s pytest.approx prošel!")

# Můžeme nastavit vlastní toleranci:
# abs = absolutní tolerance, rel = relativní tolerance
assert 3.14 == pytest.approx(3.1, abs=0.05)   # |3.14 - 3.1| = 0.04 < 0.05 → OK
print("Test s vlastní tolerancí prošel!")

### Praktický příklad: testování průměru s floaty

In [ ]:
%%writefile test_stats_floats.py
"""Tests showing float comparison pitfalls with average()."""
import pytest
from stats import average


def test_average_thirds():
    """Průměr z [1, 2] = 1.5 - toto je přesné."""
    assert average([1, 2]) == 1.5


def test_average_repeating_decimal():
    """Průměr z [1, 2, 3] = 2.0 - toto je přesné."""
    assert average([1, 2, 3]) == 2.0


def test_average_imprecise_result():
    """Průměr z [1, 2, 3, 4, 5, 6, 7] = 4.0 - toto je přesné."""
    assert average([1, 2, 3, 4, 5, 6, 7]) == 4.0


def test_average_float_inputs():
    """Průměr z desetinných čísel - tady musíme použít approx!"""
    # Průměr z [0.1, 0.2, 0.3] by měl být 0.2
    # Ale kvůli float nepřesnosti potřebujeme approx
    assert average([0.1, 0.2, 0.3]) == pytest.approx(0.2)

In [ ]:
!pytest test_stats_floats.py -v

> **Pravidlo:** Kdykoli testujete výpočet, který vrací `float`, používejte `pytest.approx()` 
> místo přímého `==`. Ušetříte si hodiny hledání "chyb", které ve skutečnosti nejsou chybami.

---

## 6. Testování výjimek — do hloubky

Výjimky nejsou chyby - jsou součástí **kontraktu** funkce. Když docstring říká
"Raises ValueError if b is zero", testujeme to stejně jako návratovou hodnotu.
Je to slib, který funkce dává uživateli.

### Základní `pytest.raises`

In [ ]:
%%writefile test_calculator_exceptions.py
"""Tests demonstrating thorough exception testing."""
import pytest
from calculator import divide


# --- Základní: ověříme, že se výjimka vyhodí ---

def test_divide_by_zero_raises_value_error():
    """Nejjednodušší forma - ověříme typ výjimky."""
    with pytest.raises(ValueError):
        divide(10, 0)


# --- Kontrola chybové zprávy pomocí 'match' ---

def test_divide_by_zero_error_message():
    """Ověříme, že zpráva výjimky obsahuje konkrétní text.
    Parametr match přijímá regulární výraz.
    """
    with pytest.raises(ValueError, match="Cannot divide by zero"):
        divide(10, 0)


# --- Přístup k objektu výjimky přes 'as' ---

def test_divide_by_zero_exception_details():
    """Pomocí 'as exc_info' získáme přístup k celému objektu výjimky.
    Můžeme zkontrolovat její atributy, typ, zprávu - cokoliv.
    """
    with pytest.raises(ValueError) as exc_info:
        divide(10, 0)

    # exc_info.value je samotný exception objekt
    assert "zero" in str(exc_info.value)
    assert exc_info.type is ValueError


# --- Ověření, že se výjimka NEVYHODÍ ---

def test_divide_valid_inputs_no_exception():
    """Důležité je testovat i to, že výjimka NENÍ vyhozena
    při validních vstupech. Stačí normální assert.
    """
    result = divide(10, 2)
    assert result == 5.0  # žádná výjimka = OK

In [ ]:
!pytest test_calculator_exceptions.py -v

### Testování vlastních (custom) výjimek

V reálném kódu si často definujeme **vlastní typy výjimek**, aby bylo jasné,
co přesně selhalo. Pytest s nimi pracuje stejně snadno:

In [ ]:
%%writefile validator.py
"""Module with custom exceptions for input validation."""


class ValidationError(Exception):
    """Custom exception for validation failures."""

    def __init__(self, field: str, message: str):
        self.field = field
        self.message = message
        super().__init__(f"{field}: {message}")


def validate_age(age: int) -> int:
    """Validate that age is a reasonable value.
    
    Raises:
        ValidationError: If age is negative or unreasonably high.
    """
    if age < 0:
        raise ValidationError("age", "must be non-negative")
    if age > 150:
        raise ValidationError("age", "unreasonably high value")
    return age

In [ ]:
%%writefile test_validator.py
"""Tests for custom exception handling."""
import pytest
from validator import validate_age, ValidationError


def test_valid_age():
    assert validate_age(25) == 25


def test_negative_age_raises_validation_error():
    """Ověříme konkrétní TYP vlastní výjimky."""
    with pytest.raises(ValidationError):
        validate_age(-5)


def test_negative_age_exception_attributes():
    """Přistoupíme k vlastním atributům výjimky (field, message)."""
    with pytest.raises(ValidationError) as exc_info:
        validate_age(-5)

    # Kontrola vlastních atributů exception objektu
    assert exc_info.value.field == "age"
    assert "non-negative" in exc_info.value.message


def test_too_old_raises_validation_error():
    with pytest.raises(ValidationError) as exc_info:
        validate_age(200)

    assert exc_info.value.field == "age"
    assert "unreasonably high" in exc_info.value.message


def test_boundary_ages_are_valid():
    """Hraniční hodnoty by měly projít bez výjimky."""
    assert validate_age(0) == 0
    assert validate_age(150) == 150

In [ ]:
!pytest test_validator.py -v

### Shrnutí technik pro testování výjimek

| Technika | Kdy použít | Příklad |
|---|---|---|
| `pytest.raises(ExType)` | Ověřit, že se výjimka vyhodí | `with pytest.raises(ValueError):` |
| `match="text"` | Zkontrolovat chybovou zprávu | `pytest.raises(ValueError, match="zero")` |
| `as exc_info` | Přistoupit k atributům výjimky | `exc_info.value.field` |
| Normální `assert` | Ověřit, že výjimka NENÍ vyhozena | `assert func(valid) == expected` |
| Custom exceptions | Specifické typy chyb s vlastními atributy | `class ValidationError(Exception)` |

---

## 7. Testování jako ověření správnosti — příklad "Bug Hunter"

Následující ukázka simuluje situaci, která se v praxi stává běžně: máte kód
(ať už ho napsal kolega, vy sami před měsícem, nebo AI) a potřebujete ověřit,
že funguje správně.

Funkce `is_palindrome` má zkontrolovat, zda je řetězec palindrom
(čte se stejně zepředu i zezadu). Podívejte se na implementaci — vypadá rozumně?

In [ ]:
%%writefile palindrome.py
"""Module with a palindrome checker - contains a subtle bug!"""


def is_palindrome(text: str) -> bool:
    """Check if the given text is a palindrome.
    
    A palindrome reads the same forwards and backwards.
    Comparison should be case-insensitive and ignore spaces.
    
    Examples:
        is_palindrome("racecar") -> True
        is_palindrome("hello")   -> False
        is_palindrome("A man a plan a canal Panama") -> True
    """
    cleaned = text.replace(" ", "")
    return cleaned == cleaned[::-1]

Na první pohled kód vypadá v pořádku. Zkusme napsat testy a uvidíme:

In [ ]:
%%writefile test_palindrome.py
"""Tests that reveal the bug in the palindrome checker."""
from palindrome import is_palindrome


# --- Happy path ---

def test_simple_palindrome():
    assert is_palindrome("racecar") == True

def test_not_a_palindrome():
    assert is_palindrome("hello") == False

def test_single_character():
    assert is_palindrome("a") == True

def test_empty_string():
    assert is_palindrome("") == True


# --- Edge cases ---

def test_palindrome_with_spaces():
    """Palindrom s mezerami - docstring říká, že se mají ignorovat."""
    assert is_palindrome("nurses run") == True

def test_palindrome_case_insensitive():
    """Docstring říká case-insensitive - tohle odhalí bug!"""
    assert is_palindrome("Racecar") == True

def test_palindrome_mixed_case_with_spaces():
    """Klasický příklad z docstringu."""
    assert is_palindrome("A man a plan a canal Panama") == True

In [ ]:
!pytest test_palindrome.py -v

### Analýza výsledků

Testy odhalily **bug**: funkce sice odstraňuje mezery, ale **neřeší velikost písmen** 
(case sensitivity), přestože docstring slibuje `case-insensitive` porovnání.

- `"Racecar"` → po odstranění mezer je pořád `"Racecar"`, ale pozpátku `"racecaR"` → **FAIL**
- `"A man a plan a canal Panama"` → `"AmanaplanacanalpanamaP"` vs `"PamanaplanacanalpanamaA"` → **FAIL**

### Oprava

Stačí přidat `.lower()` do čištění textu:

In [ ]:
%%writefile palindrome.py
"""Module with a fixed palindrome checker."""


def is_palindrome(text: str) -> bool:
    """Check if the given text is a palindrome.
    
    A palindrome reads the same forwards and backwards.
    Comparison is case-insensitive and ignores spaces.
    """
    cleaned = text.replace(" ", "").lower()  # <-- přidáno .lower()
    return cleaned == cleaned[::-1]

In [ ]:
# Po opravě by měly všechny testy projít:
!pytest test_palindrome.py -v

**Poučení:** Kód vypadal na první pohled správně, ale test na hraniční případ (velká písmena) odhalil chybu.
Klíčem bylo **čtení docstringu** — ten sliboval case-insensitive chování, ale implementace to nedělala.

> Toto je přesně situace, kde testování pomáhá při práci s AI-generovaným kódem.
> AI vám vygeneruje kód, který projde na jednoduchých příkladech,
> ale vaše testy na edge cases odhalí, co nefunguje.
> **Schopnost napsat správné testy je důležitější než schopnost napsat kód.**

---

## 8. Testování objektů (tříd)

V hodině o OOP jste se naučili vytvářet třídy. Testy tříd ověřují:

- **Inicializaci** - má objekt po vytvoření správné atributy?
- **Chování metod** - dělají metody to, co mají?
- **Změnu stavu** - mění se vnitřní stav objektu správně?
- **Chybové stavy** - vyhodí metoda chybu, když má?

### Třída `BankAccount`

In [ ]:
%%writefile bank_account.py
"""A simple bank account class for testing demonstration."""


class BankAccount:
    """Represents a bank account with basic operations."""

    def __init__(self, owner: str, balance: float = 0.0):
        """Create a new bank account.
        
        Args:
            owner: Name of the account owner.
            balance: Initial balance (default 0.0, cannot be negative).
        
        Raises:
            ValueError: If initial balance is negative.
        """
        if balance < 0:
            raise ValueError("Initial balance cannot be negative")
        self.owner = owner
        self.balance = balance

    def deposit(self, amount: float) -> None:
        """Deposit money into the account.
        
        Raises:
            ValueError: If amount is not positive.
        """
        if amount <= 0:
            raise ValueError("Deposit amount must be positive")
        self.balance += amount

    def withdraw(self, amount: float) -> None:
        """Withdraw money from the account.
        
        Raises:
            ValueError: If amount is not positive or exceeds balance.
        """
        if amount <= 0:
            raise ValueError("Withdrawal amount must be positive")
        if amount > self.balance:
            raise ValueError("Insufficient funds")
        self.balance -= amount

    def __str__(self) -> str:
        return f"BankAccount({self.owner}, balance={self.balance:.2f})"

Teď napíšeme testy. Všimněte si, jak se testy přirozeně člení podle toho, **co testují**:

In [ ]:
%%writefile test_bank_account.py
"""Tests for the BankAccount class."""
import pytest
from bank_account import BankAccount


# ===== Testy inicializace (__init__) =====

def test_create_account_default_balance():
    """Nový účet bez zadaného zůstatku má balance 0."""
    account = BankAccount("Alice")
    assert account.owner == "Alice"
    assert account.balance == 0.0


def test_create_account_with_initial_balance():
    """Nový účet se zadaným počátečním zůstatkem."""
    account = BankAccount("Bob", balance=1000.0)
    assert account.owner == "Bob"
    assert account.balance == 1000.0


def test_create_account_negative_balance_raises():
    """Vytvoření účtu se záporným zůstatkem musí selhat."""
    with pytest.raises(ValueError, match="negative"):
        BankAccount("Charlie", balance=-100.0)


# ===== Testy metody deposit =====

def test_deposit_increases_balance():
    """Vklad zvýší zůstatek o správnou částku."""
    account = BankAccount("Alice", balance=100.0)
    account.deposit(50.0)
    assert account.balance == 150.0


def test_deposit_multiple_times():
    """Více vkladů za sebou správně kumuluje zůstatek."""
    account = BankAccount("Alice")
    account.deposit(100.0)
    account.deposit(200.0)
    account.deposit(50.0)
    assert account.balance == 350.0


def test_deposit_zero_raises():
    """Vklad nulové částky musí selhat."""
    account = BankAccount("Alice", balance=100.0)
    with pytest.raises(ValueError, match="positive"):
        account.deposit(0)


def test_deposit_negative_raises():
    """Vklad záporné částky musí selhat."""
    account = BankAccount("Alice", balance=100.0)
    with pytest.raises(ValueError, match="positive"):
        account.deposit(-50.0)


# ===== Testy metody withdraw =====

def test_withdraw_decreases_balance():
    """Výběr sníží zůstatek o správnou částku."""
    account = BankAccount("Alice", balance=200.0)
    account.withdraw(50.0)
    assert account.balance == 150.0


def test_withdraw_entire_balance():
    """Výběr celého zůstatku - účet by měl mít 0."""
    account = BankAccount("Alice", balance=100.0)
    account.withdraw(100.0)
    assert account.balance == 0.0


def test_withdraw_insufficient_funds_raises():
    """Výběr vyšší částky než je zůstatek musí selhat."""
    account = BankAccount("Alice", balance=50.0)
    with pytest.raises(ValueError, match="Insufficient"):
        account.withdraw(100.0)


def test_withdraw_from_empty_account_raises():
    """Výběr z prázdného účtu musí selhat."""
    account = BankAccount("Alice")
    with pytest.raises(ValueError, match="Insufficient"):
        account.withdraw(1.0)


def test_withdraw_negative_raises():
    """Výběr záporné částky musí selhat."""
    account = BankAccount("Alice", balance=100.0)
    with pytest.raises(ValueError, match="positive"):
        account.withdraw(-10.0)


# ===== Testy kombinací operací =====

def test_deposit_then_withdraw():
    """Sekvence operací: vklad, pak výběr."""
    # ARRANGE
    account = BankAccount("Alice")

    # ACT
    account.deposit(500.0)
    account.withdraw(200.0)

    # ASSERT
    assert account.balance == 300.0

In [ ]:
!pytest test_bank_account.py -v

### Struktura testů pro třídy

Všimněte si vzoru, jak jsme testy organizovali:

| Skupina testů | Co ověřuje | Příklad |
|---|---|---|
| **Inicializace** | Správné vytvoření objektu, výchozí hodnoty, nevalidní vstupy | `test_create_account_*` |
| **Metoda deposit** | Happy path, opakované volání, nevalidní vstupy | `test_deposit_*` |
| **Metoda withdraw** | Happy path, hraniční stavy, nevalidní vstupy | `test_withdraw_*` |
| **Kombinace** | Sekvence operací, reálné scénáře | `test_deposit_then_withdraw` |

> **Tip:** U každé metody testujte zvlášť (1) normální chování, (2) hraniční případy a (3) chybové stavy.
> Každý test by měl vytvářet **svůj vlastní objekt** - testy na sobě nesmí záviset.

---

## 9. Fixtures — sdílená příprava testů

Všimli jste si, že v testech pro `BankAccount` se opakuje vytváření účtu?

```python
account = BankAccount("Alice", balance=100.0)  # tohle píšeme pořád dokola
```

**Fixture** je funkce označená dekorátorem `@pytest.fixture`, která **připraví data nebo objekty**,
které testy potřebují. Pytest ji automaticky zavolá a její výsledek předá testu jako argument.

### Proč fixtures?

1. **DRY** (Don't Repeat Yourself) - společný setup napíšete jednou
2. **Přehlednost** - test se soustředí na to, co testuje, ne na přípravu
3. **Izolace** - každý test dostane **čerstvou** instanci, testy se neovlivňují

In [ ]:
%%writefile test_bank_account_fixtures.py
"""BankAccount tests refactored with fixtures."""
import pytest
from bank_account import BankAccount


# ===== Fixtures =====

@pytest.fixture
def empty_account():
    """Fixture: účet s nulovým zůstatkem."""
    return BankAccount("Alice")


@pytest.fixture
def funded_account():
    """Fixture: účet s počátečním zůstatkem 1000 Kč."""
    return BankAccount("Bob", balance=1000.0)


# ===== Testy s fixtures =====
# Pytest vidí, že test přijímá argument se jménem fixture,
# a automaticky ji zavolá a předá výsledek.

def test_empty_account_has_zero_balance(empty_account):
    """Fixture 'empty_account' se předá jako argument."""
    assert empty_account.balance == 0.0
    assert empty_account.owner == "Alice"


def test_deposit_to_empty_account(empty_account):
    """Každý test dostane NOVOU instanci - testy se neovlivňují."""
    empty_account.deposit(500.0)
    assert empty_account.balance == 500.0


def test_funded_account_has_correct_balance(funded_account):
    assert funded_account.balance == 1000.0


def test_withdraw_from_funded_account(funded_account):
    funded_account.withdraw(300.0)
    assert funded_account.balance == 700.0


def test_withdraw_all_from_funded_account(funded_account):
    """Tato fixture má stále 1000 - předchozí test ji neovlivnil!"""
    funded_account.withdraw(1000.0)
    assert funded_account.balance == 0.0


def test_overdraft_raises(funded_account):
    """Fixtures fungují i s pytest.raises."""
    with pytest.raises(ValueError, match="Insufficient"):
        funded_account.withdraw(2000.0)

In [ ]:
!pytest test_bank_account_fixtures.py -v

### Jak fixture funguje krok za krokem

```
test_withdraw_from_funded_account(funded_account)
                                  ^^^^^^^^^^^^^^
                                  1. Pytest vidí tento parametr
                                  2. Najde fixture se stejným jménem
                                  3. Zavolá funded_account() → nový BankAccount
                                  4. Předá ho testu jako argument
                                  5. Pro KAŽDÝ test se fixture volá znovu!
```

### ⭐ Bonus: Fixture s úklidem (`yield`)

> **Poznámka:** Tato sekce je pokročilá a není vyžadována pro základní práci s testy.
> Pokud vám příjde příliš složitá, klidně ji přeskočte.

Někdy fixture potřebuje po testu **uklidit** (zavřít soubor, smazat dočasná data...).
K tomu slouží `yield` místo `return` — kód **za** yieldem se provede po skončení testu:

In [ ]:
%%writefile test_yield_fixture.py
"""Demonstration of yield fixtures with setup and teardown."""
import pytest
import os


@pytest.fixture
def temp_file():
    """Fixture, která vytvoří dočasný soubor a po testu ho smaže."""
    # --- SETUP (před yield) ---
    file_path = "test_data.txt"
    with open(file_path, "w") as f:
        f.write("test data\n")
    print(f"\n  [fixture setup] Soubor '{file_path}' vytvořen")

    yield file_path  # <-- toto se předá testu

    # --- TEARDOWN (po yield) ---
    # Tento kód se provede PO skončení testu (i když test selže!)
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f"  [fixture teardown] Soubor '{file_path}' smazán")


def test_file_exists(temp_file):
    """Test dostane cestu k souboru, fixture ho vytvořila."""
    assert os.path.exists(temp_file)


def test_file_content(temp_file):
    """Každý test dostane čerstvě vytvořený soubor."""
    with open(temp_file) as f:
        content = f.read()
    assert content == "test data\n"

In [ ]:
# -s zobrazí print výstupy (pytest je normálně skrývá)
!pytest test_yield_fixture.py -v -s

Ve výstupu vidíte, jak se pro každý test spustí **setup** (před yield) i **teardown** (po yield).
Dočasný soubor je po každém testu smazán - testy po sobě neuklízí.

### Přehled fixtures

| Varianta | Kdy použít | Klíčové slovo |
|---|---|---|
| `return` fixture | Příprava dat/objektů bez nutnosti úklidu | `return BankAccount(...)` |
| `yield` fixture | Příprava + úklid po testu (soubory, DB, síť...) | `yield path` + `os.remove(path)` |

> **Tip:** Fixture pojmenujte popisně (`empty_account`, `temp_file`) -
> název fixture se objeví v chybových zprávách pytestu a pomůže s debugováním.

---

## 10. Organizace testů v projektu

V reálném projektu testy obvykle žijí v samostatné složce. Typická struktura:

```
my_project/
├── src/                    # zdrojový kód
│   ├── calculator.py
│   └── stats.py
├── tests/                  # testy
│   ├── test_calculator.py
│   └── test_stats.py
├── requirements.txt
└── README.md
```

### Spouštění všech testů najednou

Stačí zavolat `pytest` bez argumentů - automaticky najde všechny soubory začínající na `test_`:

In [ ]:
# Spustí VŠECHNY testy v aktuální složce (a podsložkách)
!pytest -v

### Užitečné přepínače pytestu

| Příkaz | Co dělá |
|---|---|
| `pytest` | Spustí všechny testy |
| `pytest -v` | Verbose - ukáže název každého testu |
| `pytest test_calculator.py` | Spustí testy jen z jednoho souboru |
| `pytest test_calculator.py::test_add_zero` | Spustí jeden konkrétní test |
| `pytest -x` | Zastaví se po prvním selhání |
| `pytest --tb=short` | Zkrácený výpis chyb |

---

## 11. Testování výkonu (Performance Testing)

Zatím jsme testovali, **co** funkce vrátí. Ale někdy nás zajímá i **jak rychle** to udělá.
Typická otázka: *"Tato funkce musí zpracovat 10 000 prvků do 1 sekundy — splňuje to?"*

### Dva způsoby řazení

Připravíme si dvě implementace řazení — pomalou (bubble sort, O(n²))
a rychlou (vestavěný `sorted()`, O(n log n)) — a změříme rozdíl:

In [ ]:
%%writefile sorting.py
"""Sorting implementations for performance comparison."""


def bubble_sort(data: list) -> list:
    """Sort using bubble sort (O(n^2) - slow for large inputs)."""
    arr = data.copy()
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr


def builtin_sort(data: list) -> list:
    """Sort using Python's built-in sorted() (O(n log n) - fast)."""
    return sorted(data)

### Ruční měření s `time.perf_counter`

Nejjednodušší přístup — změříme čas sami a porovnáme:

In [ ]:
import time
import random
from sorting import bubble_sort, builtin_sort

# Vygenerujeme náhodný seznam
data = [random.randint(0, 10000) for _ in range(5000)]

# Měříme bubble sort
start = time.perf_counter()
result1 = bubble_sort(data)
bubble_time = time.perf_counter() - start

# Měříme built-in sort
start = time.perf_counter()
result2 = builtin_sort(data)
builtin_time = time.perf_counter() - start

# Ověříme, že oba dávají stejný výsledek
assert result1 == result2, "Oba algoritmy musí dát stejný výsledek!"

print(f"Bubble sort: {bubble_time:.4f} s")
print(f"Built-in:    {builtin_time:.6f} s")
print(f"Built-in je {bubble_time / builtin_time:.0f}× rychlejší!")

### Performance test v pytestu

Můžeme napsat test, který **selže, pokud funkce trvá příliš dlouho**.
To je užitečné, když chceme zaručit, že optimalizace nebyla omylem zpomalena:

In [ ]:
%%writefile test_sorting_performance.py
"""Performance tests - verify that functions complete within time limits."""
import time
import random
import pytest
from sorting import bubble_sort, builtin_sort


# Pomocná fixture: náhodná data pro testy
@pytest.fixture
def random_data():
    """Generate a random list of 5000 integers."""
    random.seed(42)  # fixní seed = pokaždé stejná data
    return [random.randint(0, 10000) for _ in range(5000)]


def test_builtin_sort_is_fast(random_data):
    """Built-in sort musí zvládnout 5000 prvků do 0.05 sekundy."""
    start = time.perf_counter()
    result = builtin_sort(random_data)
    elapsed = time.perf_counter() - start

    assert elapsed < 0.05, f"Trvalo {elapsed:.4f}s, limit je 0.05s"
    assert result == sorted(random_data)  # ověříme i správnost


def test_bubble_sort_is_slow(random_data):
    """Bubble sort na 5000 prvcích trvá DÉLE než 0.1 sekundy.
    Tento test ukazuje, že bubble sort je pro velká data pomalý.
    """
    start = time.perf_counter()
    result = bubble_sort(random_data)
    elapsed = time.perf_counter() - start

    assert elapsed > 0.1, f"Bubble sort byl nečekaně rychlý: {elapsed:.4f}s"
    print(f"\n  Bubble sort: {elapsed:.2f}s (očekávaně pomalý)")
    assert result == sorted(random_data)


def test_builtin_faster_than_bubble(random_data):
    """Built-in sort musí být alespoň 10× rychlejší než bubble sort."""
    # Měříme bubble sort
    start = time.perf_counter()
    bubble_sort(random_data)
    bubble_time = time.perf_counter() - start

    # Měříme built-in sort
    start = time.perf_counter()
    builtin_sort(random_data)
    builtin_time = time.perf_counter() - start

    speedup = bubble_time / builtin_time
    assert speedup > 10, f"Očekáván 10× speedup, ale bylo jen {speedup:.1f}×"
    print(f"\n  Speedup: {speedup:.0f}× (built-in je rychlejší)")

In [ ]:
!pytest test_sorting_performance.py -v -s

### Kdy použít performance testy?

| Situace | Příklad |
|---|---|
| Garantujete rychlost API | *"Endpoint musí odpovědět do 200 ms"* |
| Porovnáváte algoritmy | *"Nový algoritmus musí být alespoň 2× rychlejší"* |
| Hlídáte regresi | *"Po refactoringu nesmí být funkce pomalejší"* |

> **⚠️ Pozor:** Performance testy jsou ze své podstaty **méně stabilní** než funkční testy.
> Čas běhu závisí na zatížení počítače, operačním systému a dalších faktorech.
> Proto používejte **velkorysé limity** a neberte je jako přesná měření —
> slouží jako **ochranná síť**, ne jako benchmark.

---

## 12. Cvičení

Zkuste si nabyté znalosti procvičit na následujících úkolech.
Řešení nezahlédnete — buňky jsou prázdné a čekají na váš kód.

### Cvičení 1: Napište testy pro funkci `max_of_three`

Máte následující funkci. Napište alespoň 5 testů, které otestují
normální chování i hraniční případy:

In [ ]:
%%writefile max_three.py
"""Module with a simple function to test."""


def max_of_three(a: float, b: float, c: float) -> float:
    """Return the largest of three numbers."""
    if a >= b and a >= c:
        return a
    elif b >= a and b >= c:
        return b
    else:
        return c

In [ ]:
%%writefile test_max_three.py
"""Vaše testy pro funkci max_of_three."""
from max_three import max_of_three


# TODO: Napište testy!
# Tipy:
# - Co když jsou všechna tři čísla stejná?
# - Co když jsou dvě čísla stejná a jedno menší?
# - Co se záporními čísly?
# - Co s nulami?

def test_all_different():
    # Sem napište svůj test
    pass


In [ ]:
# Spusťte své testy:
!pytest test_max_three.py -v

### Cvičení 2: Ověřte AI-generovaný kód 🤖

Představte si, že jste požádali AI o funkci `find_longest_word(sentence)`,
která vrátí **nejdelší slovo** ve větě. AI vám dalo následující implementaci.

**Váš úkol:**
1. Napište testy (happy path + edge cases)
2. Najděte situaci, kdy funkce nefunguje správně
3. Opravte funkci

> **Hint:** Přemýšlejte, co se stane s prázdným řetězcem a s interpunkcí.

In [ ]:
%%writefile longest_word.py
"""AI-generated module - does it work correctly?"""


def find_longest_word(sentence: str) -> str:
    """Return the longest word in a sentence.
    
    If multiple words have the same length, return the first one.
    Punctuation should be ignored.
    """
    words = sentence.split()
    return max(words, key=len)

In [ ]:
%%writefile test_longest_word.py
"""Vaše testy pro find_longest_word."""
from longest_word import find_longest_word


# TODO: Napište testy!
# Co byste měli otestovat:
# - Normální věta s různě dlouhými slovy
# - Věta s interpunkcí (tečky, čárky, vykřičníky)
# - Prázdný řetězec
# - Jedno slovo
# - Více slov stejné délky

def test_basic_sentence():
    # Sem napište svůj test
    pass


In [ ]:
# Spusťte své testy:
!pytest test_longest_word.py -v

---

## 13. Shrnutí

### Co jsme se naučili

| Koncept | Popis |
|---|---|
| `assert` | Vestavěné klíčové slovo Pythonu pro kontrolní výrazy |
| **pytest** | Testovací framework — automaticky hledá a spouští testy |
| **Konvence pojmenování** | Soubory `test_*.py`, funkce `test_*()` |
| **Arrange-Act-Assert** | Vzor pro strukturování testů: připrav → proveď → ověř |
| `pytest.raises` | Ověření, že funkce vyhodí očekávanou výjimku |
| `pytest.approx` | Porovnání desetinných čísel s tolerancí (floating point) |
| **Floating point** | Proč `0.1 + 0.2 != 0.3` a jak to řešit v testech |
| **Edge cases** | Hraniční případy: prázdné vstupy, nuly, velká čísla |
| **Bug Hunter** | Testy odhalí chyby, které na první pohled nevidíte |
| **Testování tříd** | Inicializace, metody, změna stavu, chybové stavy |
| **Fixtures** | Sdílená příprava dat/objektů pro testy |

### Klíčové myšlenky

- **Testy jsou investice** — stojí čas při psaní, ale šetří hodiny při hledání chyb
- **Přemýšlejte o edge casech** — to je ta hlavní dovednost, ne syntaxe pytestu
- **Testy slouží jako dokumentace** — z testů je jasné, jak se funkce má chovat
- **Testujte chování, ne implementaci** — testujte CO funkce dělá, ne JAK to dělá
- **Pozor na floaty** — při porovnávání desetinných čísel vždy používejte `pytest.approx()`
- **V éře AI: testy > kód** — schopnost napsat dobré testy je cennější než schopnost napsat kód

### Zdroje

- [pytest dokumentace](https://docs.pytest.org/) — oficiální docs
- [Real Python: Testing in Python](https://realpython.com/python-testing/) — podrobný tutoriál
- [Arrange-Act-Assert pattern](https://automationpanda.com/2020/07/07/arrange-act-assert-a-pattern-for-writing-good-tests/) — detailnější vysvětlení AAA
- [Floating Point Arithmetic](https://docs.python.org/3/tutorial/floatingpoint.html) — Python docs o float problematice

---

## Úklid

Následující buňka smaže pomocné soubory vytvořené v tomto notebooku:

In [ ]:
import os

for f in [
    "calculator.py", "test_calculator.py", "test_calculator_fail.py",
    "test_calculator_aaa.py", "test_calculator_exceptions.py",
    "stats.py", "test_stats.py", "test_stats_floats.py",
    "bank_account.py", "test_bank_account.py",
    "test_bank_account_fixtures.py", "test_yield_fixture.py",
    "validator.py", "test_validator.py",
    "palindrome.py", "test_palindrome.py",
    "max_three.py", "test_max_three.py",
    "longest_word.py", "test_longest_word.py",
    "sorting.py", "test_sorting_performance.py",
]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Smazán: {f}")

# Smaže pytest cache
import shutil
if os.path.exists(".pytest_cache"):
    shutil.rmtree(".pytest_cache")
    print("Smazán: .pytest_cache/")
if os.path.exists("__pycache__"):
    shutil.rmtree("__pycache__")
    print("Smazán: __pycache__/")